## Run This Line of Code

In [ ]:
#uv add "pygam>=0.11" "scipy>=1.16.3,<1.17" --upgrade-package pygam

## OR

In [ ]:
#%pip install --upgrade "pygam>=0.11"

# GAMs


## General Set-UP

In [31]:
import numpy as np
import pandas as pd
from scipy.special import expit
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.metrics import mean_squared_error, accuracy_score, mean_absolute_error
from palmerpenguins import load_penguins
import statsmodels.api as sm

## Generate Data

In [ ]:
rng = np.random.default_rng(909)


def simulate_quartic_data(n=1000, x_sd=0.25, noise_sd=0.5):
    """Simulate data from the quartic relationship used in the slides."""
    x = rng.normal(loc=0, scale=x_sd, size=n)
    y = (
        1
        + 5.3 * x
        - 45 * x**2
        - 35.5 * x**3
        + 60 * x**4
        + rng.normal(loc=0, scale=noise_sd, size=n)
    )
    return pd.DataFrame({"x": x, "y": y})

motivating_polynomial_data = simulate_quartic_data()

n = 500
n_trials = 1  # Bernoulli outcomes, suitable for LogisticGAM

# Simulate one predictor
x = rng.uniform(0, 2 * np.pi, size=n)

# Transform the predictor and compute probabilities
log_odds = -0.5 + 2 * np.sin(x)
probability = expit(log_odds)

# Simulate binomial outcomes
y = rng.binomial(n=n_trials, p=probability)

# Store the simulated data
simulated_binomial_data = pd.DataFrame({
    "x": x,
    "y": y,
    "true_probability": probability
})



x_gam = rng.normal(loc=0, scale=1, size=1000)
y_gam = rng.normal(loc=4, scale=1, size=1000)
mu_gam = np.exp(-2 + expit(x_gam) + np.sqrt(y_gam + 15))
z_gam = rng.poisson(mu_gam)
poisson_gam_data = pd.DataFrame(
    {"x": x_gam, "y": y_gam, "mu": mu_gam, "z": z_gam}
)





## GAMs


In [ ]:
from pygam import LinearGAM, s
from sklearn.metrics import mean_squared_error, r2_score



## Data


In [ ]:
penguins = load_penguins().dropna()
penguins.columns

## Regression

### Data Prep

In [ ]:
mass_features = ["bill_length_mm", "bill_depth_mm",  
                 "flipper_length_mm"]
X_mass = (
    pd.get_dummies(penguins[mass_features], dtype = int, drop_first=True)
) 
y_mass = penguins["body_mass_g"]



X_train, X_test, y_train, y_test = train_test_split(
    X_mass, y_mass,
    test_size=0.20,
    random_state=909
)

### Fit Model

In [ ]:

# Convert pandas objects to NumPy arrays
X_train_gam = X_train.to_numpy(dtype=float)
X_test_gam = X_test.to_numpy(dtype=float)
y_train_gam = y_train.to_numpy(dtype=float)

# Specify one penalized spline for each numerical predictor
gam = LinearGAM(
    s(0, basis="ps", n_splines=20, spline_order=3)
    + s(1, basis="ps", n_splines=20, spline_order=3)
    + s(2, basis="ps", n_splines=20, spline_order=3)
)

# Fit the model and select the smoothing penalty using GCV
gam.gridsearch(
    X_train_gam,
    y_train_gam,
    lam=np.logspace(-3, 5, 30),
    objective="GCV",
    progress=False
)

# Predict body mass for the test observations
y_pred = gam.predict(X_test_gam)


### Prediction Error

In [ ]:
mean_squared_error(y_test, y_pred)

## Classification

In [ ]:
from pygam import LogisticGAM, s
from sklearn.metrics import accuracy_score

### Data Prep

In [ ]:
penguins["gentoo"] = np.where(
    penguins["species"] == "Gentoo",
    "Gentoo",
    "Other",
)
penguins["gentoo_binary"] = (
    penguins["species"] == "Gentoo"
).astype(int)


gentoo_features = ["bill_length_mm", "bill_depth_mm"]
gentoo_features = ["bill_length_mm", "bill_depth_mm"]
X_gentoo = penguins[gentoo_features]
y_gentoo = penguins["gentoo_binary"]

# Split once; stratify preserves the class proportions
X_train, X_test, y_train, y_test = train_test_split(
    X_gentoo,
    y_gentoo,
    test_size=0.20,
    random_state=909,
    stratify=y_gentoo,
)
X_train_gam = X_train.to_numpy(dtype=float)
X_test_gam = X_test.to_numpy(dtype=float)
y_train_gam = y_train.to_numpy(dtype=int)


### Fitting Model

In [ ]:
gam = LogisticGAM(
    s(0, basis="ps", n_splines=10, spline_order=3)
    + s(1, basis="ps", n_splines=10, spline_order=3),
    max_iter=10000,
)

# Select the smoothing penalty using training data
gam.gridsearch(
    X_train_gam,
    y_train_gam,
    lam=np.logspace(-3, 5, 30),
    objective="UBRE",
    progress=False,
)

# Predicted probabilities of Gentoo
y_prob = gam.predict_proba(X_test_gam)

# Classify using a probability threshold of 0.50
y_pred = (y_prob >= 0.50).astype(int)



### Classification Error

In [ ]:
# Evaluate on the test set
accuracy_score(y_test, y_pred)


## Poisson GAM

In [26]:
from pygam import PoissonGAM, s

### Data Prep

In [24]:
X_pois = poisson_gam_data[["x", "y"]]
y_pois = poisson_gam_data[["y"]]

X_train, X_test, y_train, y_test = train_test_split(
    X_pois,
    y_pois,
    test_size=0.20,
    random_state=909
)
X_train_gam = X_train.to_numpy(dtype=float)
X_test_gam = X_test.to_numpy(dtype=float)
y_train_gam = y_train.to_numpy(dtype=int)

### Fit Model

In [32]:
gam = PoissonGAM(
    s(0, basis="ps", n_splines=10)
    + s(1, basis="ps", n_splines=10)
)

gam.gridsearch(
    X_train_gam,
    y_train_gam,
    lam=np.logspace(-3, 5, 30),
    objective="UBRE",
    progress=False,
)

# Predicted expected counts; these can be nonintegers
y_pred = gam.predict(X_test_gam)

# MSE may not be appropriate for Poisson GAM; No neead to exponentiate predictions
mean_absolute_error(y_pred, y_test)

0.5163766724634266

## Other Types of GAMs

| Name | Function / Purpose | Distribution | Default Link |
|---|---|---|---|
| `LinearGAM()` | Models a continuous response, such as body mass | Normal (Gaussian) | Identity |
| `LogisticGAM()` | Models a binary response, such as Gentoo versus Other | Binomial | Logit |
| `PoissonGAM()` | Models nonnegative integer counts, such as observations per survey | Poisson | Log |
| `GammaGAM()` | Models strictly positive, continuous responses, such as costs or durations | Gamma | Log |
| `InvGaussGAM()` | Models strictly positive, continuous responses with substantial right skew | Inverse Gaussian | Log |
| `GAM()` | General model class allowing explicit specification of the distribution and link | User-specified; Normal by default | User-specified; Identity by default |
